# 9.3 策略一：GS1 不合轴 + 切 S2 负载均衡

本章第一个完整分核策略：**保持 B、N 嵌套循环，S1（GS1）作为最内层迭代轴贪心累加，同时在 S2 方向也允许切分**。

## 1. 先明确术语

- **GS1（global S1）**：单个 (b, n) 内的 Q 行方向。所谓「不合轴」，即分核循环保持 `for b → for n → for s1` 的**嵌套结构**，而不是把 B×N×S1 展平；
- **切 S2（flash decoding）**：当 Q 短 KV 长（decode 场景 Q_length 常为 1，KV 累积上万）时，仅靠 B×N×S1 并行度严重不足，**必须把 KV 方向也切给不同核**。

## 2. 为什么 decode 必须切 S2

算一笔并行度账：

```
decode 场景: B=1, N=8, S1=1     → B×N×S1 = 8 个任务
可用核数: 40
→ 32 个核空转！
```

S1 方向只有 1 行 Q，再怎么贪心也切不出更多任务。唯一的并行度来源是 S2：把 KV 切成 40 段，每核算一段的部分注意力，最后 merge。

## 3. 不合轴贪心分核的完整流程

循环骨架（对应素材 `ComputeSplitNBSeq` 的结构）：

```cpp
for (b : B)                      // 外层 batch
    for (n : N)                  // 中层 head
        for (s1 : S1BlocksOf(b)) // 内层 GS1, 逐行
            curWeight += GetSInnerBlockNums(b, s1);   // 该行有效 S2 块数
            dif = target * (curCore + 1) - curWeight;  // 距理想负载的差
            if (sInnerBlockNums - dif > dif) {         // 超过容差 → 封核
                记录当前核的 [b, n, s1] 结束边界;
                curCore++;
                记录新核的起始边界;
            }
```

**封核判据的直觉**：`dif` 是「离理想负载还差多少」。当下一行的块数比缺口还大（`sInnerBlockNums > 2·dif`），说明再塞这行会「跨过」理想值太多——不如让它去下一个核，两边各自更接近理想值。

分核结果用**六个数组**记录：每核的 S1 起止、B 起止、N 起止（kernel 侧据此定位自己的任务区间）。

## 4. 切 S2 之后的维度补齐

不合轴策略里切 S2 发生在两个位置：

| 位置 | 做法 | 输出 |
|--|--|--|
| **核间切 S2**（本节） | 分核循环里 S2 方向也按有效块数参与装箱 | 每核一段 KV + 一份部分结果 |
| **softmax 归并**（[9.5](09.05_s2_merge.ipynb)） | merge kernel 把各核部分结果合成最终 attention | 全局 softmax 正确性 |

## 图示

![GS1 合轴 vs 不合轴](images/gs1_axis.svg)

## 动手实验：不合轴贪心分核（带 section）


In [ ]:
# 动手：模拟 ComputeSplitNBSeq —— 不合轴 + 因果 mask + 变长 section
import numpy as np

n_cores = 8
S1_BASE, S2_BASE = 128, 128

# 带变长 section 的输入: 每个 batch 的有效 Q 长度与 KV 长度
B = 4
actual_s1  = [128, 512, 256, 384]      # 每 batch 有效 Q
actual_kv  = [1280, 4096, 1152, 2048]   # 每 batch 有效 KV（含 shared prefix）
N = 2

# 构造所有 (b, n, s1) 行的有效权重
rows = []                                # (b, n, s1_idx, weight)
for b in range(B):
    s1_blocks = actual_s1[b] // S1_BASE
    kv_blocks = actual_kv[b] // S2_BASE
    for n in range(N):
        for s1 in range(s1_blocks):
            # 因果 mask: 第 s1 行的有效 S2 块数
            valid = (s1 + 1) * kv_blocks // s1_blocks
            rows.append((b, n, s1, max(valid, 1)))

total = sum(w for *_, w in rows)
target = total / n_cores
print(f"总有效块数 {total}, 每核期望 {target:.1f}, 行数 {len(rows)}")

# ---- 不合轴贪心: B、N 嵌套顺序遍历（rows 已按此顺序构造）----
def greedy_split(items, n_cores, tol=0.0):
    total = sum(w for *_, w in items)
    target = total / n_cores
    cores, cur, cur_w = [], [], 0.0
    for it in items:
        w = it[3]
        dif = target * (len(cores) + 1) - cur_w          # 距理想负载缺口
        if cur and (w - dif > dif):                       # 封核判据
            cores.append(cur); cur = []; cur_w = 0.0
            dif = target * (len(cores) + 1) - cur_w
        cur.append(it); cur_w += w
    if cur: cores.append(cur)
    return cores

cores = greedy_split(rows, n_cores)
loads = [sum(w for *_, w in c) for c in cores]
print(f"\n分到 {len(cores)} 核, 各核负载: {loads}")
print(f"最大/最小 = {max(loads)/min(loads):.2f}, 理想 = {target:.1f}")
print(f">>> 负载最大相对偏差 {(max(loads)-target)/target:.1%}")


### 观察结论

贪心装箱后各核负载已相当接近理想值——**这正是「按有效块数而非名义 shape 分配」的收益**。剩余偏差来自行粒度：一行 32 块的权重没法劈开，只能整行给一个核。要进一步压偏差，就得把切分粒度变细——这正是 9.4 合轴方案的动机。

## 5. 切 S2 的分核扩展

上面每行仍是「整行归属一个核」。decode 场景行数太少，需要**把行内的 S2 段也当作独立装箱单元**：

```
装箱单元从 (b, n, s1) 行
          变成 (b, n, s1, s2段) 二维块
```

每个 (b,n,s1) 行按 `s2Base` 切成若干段，段作为独立任务装箱——并行度上限从 `行数` 提到 `总有效块数`。代价是每段只产出**部分 softmax 结果**，必须经过 merge（9.5）。

## 6. 不合轴方案的优缺点

| 优点 | 缺点 |
|--|--|
| 循环结构与 kernel 侧天然对齐（B/N 定位简单） | 封核边界只能落在 (b,n,s1) 行边界，粒度粗 |
| 六数组结果紧凑，kernel 消费方便 | 极端不均衡时（单行权重远超 target）无法劈行 |
| Host 侧实现直接，改动小 | 切 S2 需要额外的 merge 通路 |

## 本节要点

- 不合轴 = 保持 `for b → for n → for s1` 嵌套，GS1 是最内层贪心轴；
- 封核判据 `sInnerBlockNums - dif > dif`：下一行会跨过理想值太多就让它去下一个核；
- decode 场景（Q 短 KV 长）并行度只能靠切 S2，装箱单元细化到 (行, S2段)；
- 六数组记录分核结果，kernel 侧据此定位任务区间。

## 小测验

1. decode 场景为什么必须切 S2？用并行度公式说明。
2. 写出封核判据并解释它的直觉。
3. 切 S2 后每个核的输出是什么？为什么必须 merge？
4. 不合轴方案的主要局限是什么？它如何引出合轴方案？

>>> 参考答案见 [answer/09.03_answer.txt](answer/09.03_answer.txt)
